# Notebook 3 — General workflow: screening → refinement → web

The project has two hydrodynamic engines behind one output contract. This notebook walks the full pipeline and the commands that drive it. Detailed stages are in `../architecture/WORKFLOW.md`.


## Learning objectives

- Explain the two-engine / one-contract design.
- Name the stages: screening, hotspot clustering, TELEMAC cases, post-processing, web serving.
- Know the CLI entry points for each stage.


## 3.1 The pipeline

```
screening (python)  -->  hotspots.geojson
        |
        v
cluster hotspots  -->  cases/region-001/{mesh.slf, mesh.cli, mesh.liq, case.cas}
        |
        v  (docker run flussplan/telemac telemac2d.py case.cas)
TELEMAC-2D  -->  cases/region-001/r2d.slf
        |
        v
postprocess  -->  output/telemac/region-001/{results.nc, *.tif, hotspots.geojson}
        |
        v
web (Flask + MapLibre) serves screening and refinement alike
```


![The two-engine pipeline: screening, TELEMAC refinement, post-processing and web serving.](images/fig_pipeline.png)


## 3.2 Stage 1 — screening

The Python solver runs over the whole Philippine bounding box and writes the canonical outputs, including `hotspots.geojson`:

```bash
python -m src.model.run
```

Override defaults with CLI flags, e.g. `--duration-days 15`, `--resolution-km 2.0`, `--output-dir output`, or `--engine telemac2d`.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

cwd = next((c for c in (Path('.'), Path('../..')) if (c / 'src').is_dir()),
           Path('.'))
env = dict(os.environ, PYTHONPATH=str((cwd / 'src').resolve()))
try:
    r = subprocess.run([sys.executable, '-m', 'src.model.run', '--help'],
                       capture_output=True, text=True, timeout=60,
                       cwd=cwd, env=env)
    print(r.stdout or r.stderr)
except Exception as exc:
    print('screening CLI not runnable here:', exc)


usage: run.py [-h] [--config CONFIG] [--output-dir OUTPUT_DIR]
              [--duration-days DURATION_DAYS] [--resolution-km RESOLUTION_KM]
              [--tidal-source TIDAL_SOURCE] [--resume RESULTS_NC]
              [--engine ENGINE]

Tidal hydrodynamic screening model

options:
  -h, --help            show this help message and exit
  --config, -c CONFIG   Path to config YAML
  --output-dir, -o OUTPUT_DIR
                        Override output directory
  --duration-days DURATION_DAYS
                        Override simulation duration
  --resolution-km RESOLUTION_KM
                        Override grid resolution
  --tidal-source TIDAL_SOURCE
                        Override tidal forcing source: synthetic | got |
                        fes2014 | tpxo9
  --resume RESULTS_NC   Continue from the last snapshot in an existing
                        results.nc
  --engine ENGINE       Override engine: python (default) | telemac2d



## 3.3 Stage 2 — hotspot clustering and cases

Screening hotspots are scattered points; a refinement needs a contiguous sub-domain. `cluster_hotspots` groups them (highest-power-first, greedy by great-circle distance) and expands each cluster with a margin. Explicit strait sites in `telemac2d.mesh.boundary.sites` take precedence over auto-clustering.

```bash
python -m model.telemac prepare --cases-dir cases
```

Each region becomes a self-contained case directory with mesh, boundary, liquid-boundary, and steering files plus manifests.


## 3.4 Stage 3 — TELEMAC-2D run

TELEMAC runs **only** inside a pinned public Docker image (`flussplan/telemac:v8-latest` by default). The runner mounts the case directory and invokes `telemac2d.py case.cas`:

```bash
python -m model.telemac run --case cases/region-001
```


In [2]:
cwd = next((c for c in (Path('.'), Path('../..')) if (c / 'src').is_dir()),
           Path('.'))
env = dict(os.environ, PYTHONPATH=str((cwd / 'src').resolve()))
try:
    r = subprocess.run([sys.executable, '-m', 'model.telemac', '--help'],
                       capture_output=True, text=True, timeout=60,
                       cwd=cwd, env=env)
    print(r.stdout or r.stderr)
except Exception as exc:
    print('telemac CLI not runnable here:', exc)


usage: model.telemac [-h] [--config CONFIG]
                     {prepare,run,postprocess,pipeline} ...

TELEMAC-2D refinement backend

positional arguments:
  {prepare,run,postprocess,pipeline}
    prepare             cluster hotspots and write cases
    run                 run prepared case(s)
    postprocess         convert .slf to canonical outputs
    pipeline            prepare + run + postprocess

options:
  -h, --help            show this help message and exit
  --config, -c CONFIG   Path to config YAML



## 3.5 Stage 4 — post-processing

`postprocess_case` rasterises the unstructured node fields onto a regular lon/lat grid and writes the **same canonical products**, tagged as `source=telemac2d`:

```bash
python -m model.telemac postprocess --case-dir cases/region-001 \
    --output-dir output/telemac/region-001
```

It also writes `reconciliation.json`, comparing the refinement with the parent screening in the same bounding box (see `../engines/RECONCILIATION.md`).


In [3]:
for root in (Path('output/telemac'), Path('../../output/telemac')):
    if root.is_dir():
        for region in sorted(r for r in root.iterdir() if r.is_dir()):
            files = sorted(p.name for p in region.iterdir() if p.is_file())
            print(f'{region.name}: {files}')
        break


region-001: ['bathymetry.tif', 'distance_to_coast.tif', 'hotspots.geojson', 'max_current_speed.tif', 'reconciliation.json', 'results.nc', 'tidal_power_density.tif']
region-002: ['bathymetry.tif', 'distance_to_coast.tif', 'hotspots.geojson', 'max_current_speed.tif', 'reconciliation.json', 'results.nc', 'tidal_power_density.tif']
region-003: ['bathymetry.tif', 'distance_to_coast.tif', 'hotspots.geojson', 'max_current_speed.tif', 'reconciliation.json', 'results.nc', 'tidal_power_density.tif']


## 3.6 Stage 5 — web serving

Because the output contract is identical, the Flask/MapLibre app serves screening and refinement outputs unchanged:

```bash
docker compose up -d --build    # http://localhost:8001
```

`GET /api/datasets` lists the screening dataset plus every TELEMAC region; dataset-aware endpoints accept `?region=region-001`.


## Next

The pipeline is clear. Move to [Notebook 4 — model](4.model.ipynb) for a hands-on screening run with the actual Python APIs.

---

[Index](README.md) · [← 2.data.ipynb](2.data.ipynb) · [4.model.ipynb →](4.model.ipynb)
